In [47]:
import sys
import os
import pandas as pd
import importlib
import src.features as F
import pyarrow as pa
import pyarrow.parquet as pq


from dotenv import load_dotenv
from pathlib import Path

%pip install pyarrow
%pip install duckdb fastparquet



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Repo root

In [48]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

Repo root: /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction


## loading dataset

In [49]:
import os
import pandas as pd
from pathlib import Path

# 1) Where the data lives
DATA_DIR = Path(os.environ.get("AQI_DATA_DIR", REPO / "data"))
D1 = DATA_DIR / "station_hour_merged.parquet"
print("D1 is:", D1)
assert D1.exists(), f"Cannot find {D1}. Folder contains: {sorted(p.name for p in DATA_DIR.iterdir())}"

KEEP = ["StationId", "City", "Datetime", "AQI",
        "PM2.5", "PM10", "NO2", "NOx", "NH3", "CO", "SO2", "O3",
        "Temperature_C", "Relative_Humidity_pct",
        "Wind_Speed_kmh", "Wind_Direction_deg"]


def load_parquet(path, columns):
    """Try several parquet readers and return the first DataFrame that loads."""
    errors = {}

    # Attempt 1: pandas + pyarrow
    try:
        return pd.read_parquet(path, columns=columns, engine="pyarrow"), "pyarrow"
    except Exception as e:
        errors["pyarrow"] = f"{type(e).__name__}: {str(e)[:150]}"

    # Attempt 2: pandas + fastparquet (a different reader, no pyarrow)
    try:
        return pd.read_parquet(path, columns=columns, engine="fastparquet"), "fastparquet"
    except Exception as e:
        errors["fastparquet"] = f"{type(e).__name__}: {str(e)[:150]}"

    # Attempt 3: DuckDB (independent of pyarrow)
    try:
        import duckdb
        cols = ", ".join(f'"{c}"' for c in columns)
        p = str(path).replace("'", "''")
        return duckdb.sql(f"SELECT {cols} FROM read_parquet('{p}')").df(), "duckdb"
    except Exception as e:
        errors["duckdb"] = f"{type(e).__name__}: {str(e)[:150]}"

    for name, msg in errors.items():
        print(f"  {name} failed -> {msg}")
    raise RuntimeError("No parquet reader worked. See the messages above.")


df, reader = load_parquet(D1, KEEP)
print("loaded with:", reader)

df = df.sort_values(["StationId", "Datetime"]).reset_index(drop=True)

# 2) Sanity checks against what we were told about D1
print("shape:         ", df.shape)                                   # expect (2589083, 16)
print("stations:      ", df["StationId"].nunique())                  # expect 110
print("cities:        ", df["City"].nunique())                       # expect 26
print("date range:    ", df["Datetime"].min(), "->", df["Datetime"].max())
print("AQI missing %: ", round(df["AQI"].isna().mean() * 100, 1))    # expect about 22
print("Datetime type: ", df["Datetime"].dtype)                       # expect datetime64[...]
print("wind dir range:", df["Wind_Direction_deg"].min(), "-", df["Wind_Direction_deg"].max())  # expect 1 - 360

D1 is: /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction/data/station_hour_merged.parquet
loaded with: fastparquet
shape:          (2589083, 16)
stations:       110
cities:         26
date range:     2015-01-01 01:00:00 -> 2020-07-01 00:00:00
AQI missing %:  22.0
Datetime type:  datetime64[ns]
wind dir range: 1 - 360


In [ ]:
importlib.reload(F)
F.assert_contiguous(df)

OK: 1 station(s), 3,320 rows, hourly-contiguous within every station.


In [51]:
df_one = df[df["StationId"] == "KL007"].copy().reset_index(drop=True)
feat_one = F.build_features(df_one)

print("rows before:", len(df_one), "| rows after:", len(feat_one))
new_cols = [c for c in feat_one.columns if c not in df_one.columns]
print(new_cols)
print(feat_one[new_cols].isna().sum())

AttributeError: module 'src.features' has no attribute 'build_features'

In [52]:
# Cell 4: test build_features on one station (KL007)

# Guard: confirm the module Cell 3 loaded actually has the function
if not hasattr(F, "build_features"):
    raise RuntimeError(
        f"build_features not found in {F.__file__}. "
        f"Functions there: {[n for n in dir(F) if not n.startswith('_')]}. "
        "Re-run Cell 3, and check that src/features.py is saved."
    )

df_one = df[df["StationId"] == "KL007"].copy().reset_index(drop=True)
feat_one = F.build_features(df_one)

print("rows before:", len(df_one), "| rows after:", len(feat_one))

new_cols = [c for c in feat_one.columns if c not in df_one.columns]
print(len(new_cols), "new columns:", new_cols)
print(feat_one[new_cols].isna().sum())

# Leakage spot-check: Target_AQI_24h must equal the AQI looked up 24h later
lookup = df_one.set_index(["StationId", "Datetime"])["AQI"]
print("\nspot-check (last two values on each line must match):")
for _, r in feat_one.sample(5, random_state=0).iterrows():
    t24 = lookup.get((r["StationId"], r["Datetime"] + pd.Timedelta(hours=24)))
    print(r["Datetime"], "| Target_AQI_24h:", r["Target_AQI_24h"], "| direct lookup:", t24)

RuntimeError: build_features not found in /Users/gokulvs/ICT.projects/AQI_reports/AQI-Air-_pollution_prediction/src/features.py. Functions there: ['HOUR', 'assert_contiguous', 'pd']. Re-run Cell 3, and check that src/features.py is saved.